# Step 2: Process Features

## Traitement des attributs

In [ ]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os

from method_a_buffer import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_pedestrian_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")



In [ ]:

# Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()
print('Boucle sur chaque attribut... peut prendre du temps (5-10mn)')

# Boucle sur le derniers attributs
for _, row in attributs_info.iterrows():
    if row['include_in_index']:
        attribute_name = row['attribute']
        method = row['method']
        how = row['how']
        value_column = row['value_column']
        buffer_size = row['buffer_size']
        geometry_type = row['geometry_type']
        feature_query_expr = make_feature_query(
        row.get('filter_column'),
        row.get('filter_values')
    )

        # Charger la couche attribut depuis gpd_attributs
        attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

        # Appliquer la méthode
        if method == "A": # Buffer feature extraction
            attribute_df = extract_buffer_feature(
                segments_gdf = segmented_net,
                feature_gdf = attribute_gdf,
                feature_name=attribute_name,
                geom_kind=geometry_type,
                buffer_radius=buffer_size,
                how=how,
                value_column=value_column,
                crs_meter_epsg=operation_crs,
                feature_query  = feature_query_expr,
            )
            print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
                # Debug duplicates
            if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
                print("\nDEBUG: Found duplicate segment assignments")
                dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
                print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
                # Keep only the first occurrence for each segment_id
                attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
                print("Dropped duplicates, keeping first occurrence")
            print(f"Spatial join computed for {attribute_name} with method {method}") 
        
        # Ajoute d'autres méthodes si besoin


        # Ajouter la colonne au GeoDataFrame principal
        segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)

# Sauvegarder
segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


In [ ]:

segmented_net.head(20)


In [ ]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
